# 06a · Separating a sentence into situation and phrase

The custom mechanism (situation classifier + phrase classifier + rule) needs the two parts of a sentence. In
dataset v2 they were given; in real tweets they are not, e.g.

> *Love waiting 2 hours for the bus* → phrase **Love**, situation **waiting 2 hours for the bus**.

Three segmenters (all may answer "not found" for a part):

| | segmenter | size | training |
|---|---|---|---|
| a | **rule**: sentiment cues (VADER lexicon + a short list of multi-word cues) and punctuation | 0 | none, fully transparent |
| b | **extractive QA** (DistilBERT-SQuAD): *What happened?* / *How does the writer feel about it?* | 66 M | none |
| c | **token classifier** (DistilBERT): every word → O / situation / phrase | 66 M | synthetic sentences |

**Synthetic sentences:** the 140 situations of v2 in ~30 templates (phrase before, after or on both sides of the
situation, plus sentences with only one part). The test part uses templates, situations and phrases that the
classifier never saw.

**Evaluation:** (1) synthetic test sentences; (2) 50 tweets of the iSarcasmEval test set annotated by hand
(`data/isarcasm_annotated_50.csv`: 25 sarcastic, 25 not). Finally all segmenters are applied to the whole
iSarcasmEval test set (1400 tweets) and to the 200 pairs of task C; the result is saved for 06b.

**Runtime:** L4 / T4 ≈ 10 min (training ≈ 2–3 min). CPU works but training takes ≈ 30 min.

## Setup

In [ ]:
import os, sys
REPO = "https://github.com/vasilis-pantzos/sarcasm-proposal.git"
ROOT = "/content/sarcasm-proposal"
if not os.path.exists(ROOT):
    !git clone -q {REPO} {ROOT}
else:
    !git -C {ROOT} pull -q
os.chdir(ROOT)
sys.path.insert(0, ROOT)
%pip install -q vaderSentiment
ISARC = "/content/iSarcasmEval"
if not os.path.exists(ISARC):
    !git clone -q https://github.com/iabufarha/iSarcasmEval.git {ISARC}

In [ ]:
import os, json, time, numpy as np, pandas as pd, torch
from IPython.display import display
from src.prompts import load_dataset
from src.segment import (make_synthetic, evaluate, labels_from_spans, words, LexiconRule, QASegmenter,
                         TokenSegmenter, load_isarcasm, CATEGORIES)
from src.utils import mount_drive, results_dir, cached, set_seed

# ---------------- settings ----------------
N_PER_TEMPLATE = 60      # synthetic sentences per training template
EPOCHS = 3
SEED = 0
# ------------------------------------------
SUB = "06a_segmenter"
mount_drive()
print("results go to:", results_dir(SUB), "| GPU:", torch.cuda.is_available())
set_seed(SEED)

## 1 · Synthetic sentences

In [ ]:
v2 = load_dataset("v2")
SYN = make_synthetic(v2, v2, n_per_template=N_PER_TEMPLATE, seed=SEED)
print(SYN.groupby(["split", "family"]).size().unstack(), "\n")
print("templates:", SYN.groupby("split").template.nunique().to_dict(),
      "| situations:", SYN.groupby("split").situation.nunique().to_dict())
for t, lab in SYN.sample(10, random_state=1)[["text", "labels"]].values:
    ws = [w for w, _, _ in words(t)]
    print(" ".join(f"[{w}]" if l == "PHR" else (f"<{w}>" if l == "SIT" else w) for w, l in zip(ws, lab)))
print("\n[phrase]  <situation>")
SYN.drop(columns=["labels"]).to_csv(os.path.join(results_dir(SUB), "synthetic.csv"), index=False)
TR, TE = SYN[SYN.split == "train"], SYN[SYN.split == "test"]

## 2 · The three segmenters

In [ ]:
SEG = {"a · rule": LexiconRule()}

t0 = time.time()
qa = QASegmenter()
fit = TR.sample(min(400, len(TR)), random_state=SEED)
acc = qa.fit_threshold(fit.text, fit.has_both)
SEG["b · QA"] = qa.batch
print(f"QA model loaded ({time.time() - t0:.0f} s); minimum answer score chosen on training sentences: "
      f"{qa.min_score} (accuracy of 'both found': " + ", ".join(f"{k}: {v:.2f}" for k, v in acc.items()) + ")")

path = os.path.join(results_dir(SUB), "token_classifier")
if os.path.exists(os.path.join(path, "config.json")):
    tc = TokenSegmenter(path)
    print("token classifier loaded from", path)
else:
    tc = TokenSegmenter()
    t0 = time.time()
    tc.train(TR.text.tolist(), TR.labels.tolist(), epochs=EPOCHS, seed=SEED)
    tc.save(path)
    print(f"token classifier trained in {(time.time() - t0) / 60:.1f} min, saved to", path)
SEG["c · token classifier"] = tc.batch

def run(name, texts):
    f = SEG[name]
    return f(list(texts)) if name != "a · rule" else [f(t) for t in texts]

## 3 · Evaluation on synthetic sentences (unseen templates, situations and phrases)

Word-level precision / recall / F1 for the situation and the phrase; "both found" compares whether a segmenter
finds both parts with whether the sentence has both.

In [ ]:
rows, rows_f = {}, []
for name in SEG:
    pred = run(name, TE.text)
    rows[name] = evaluate(TE.text.tolist(), TE.labels.tolist(), pred)
    for fam in TE.family.unique():
        m = (TE.family == fam).values
        e = evaluate(TE.text[m].tolist(), TE.labels[m].tolist(), [p for p, k in zip(pred, m) if k])
        rows_f.append({"segmenter": name, "family": fam, "SIT F1": e["SIT F1"], "PHR F1": e["PHR F1"],
                       "both found: accuracy": e["both found: accuracy"]})
SYN_EVAL = pd.DataFrame(rows).T
display(SYN_EVAL.round(3))
display(pd.DataFrame(rows_f).pivot(index="family", columns="segmenter").round(2))

## 4 · Evaluation on 50 hand-annotated tweets

Annotation (`data/isarcasm_annotated_50.csv`): the phrase and the situation as exact substrings, empty when the
tweet has no such structure (e.g. satire, rhetorical questions, irony between two claims). Only 13 of the 25
sarcastic tweets have the structure — a first sign of how much of real sarcasm is "positive words about a
negative situation".

In [ ]:
ANN = pd.read_csv("data/isarcasm_annotated_50.csv").fillna("")
def gold(t, s, p):
    return labels_from_spans(t, {"SIT": [(t.index(s), t.index(s) + len(s))] if s else [],
                                 "PHR": [(t.index(p), t.index(p) + len(p))] if p else []})
G = [gold(t, s, p) for t, s, p in zip(ANN.tweet, ANN.situation, ANN.phrase)]
print("tweets with both parts:", int(((ANN.situation != "") & (ANN.phrase != "")).sum()), "of", len(ANN),
      "| among the sarcastic:", int(((ANN.situation != "") & (ANN.phrase != "") & (ANN.sarcastic == 1)).sum()), "of 25")
ANN_PRED, rows = {}, {}
for name in SEG:
    ANN_PRED[name] = run(name, ANN.tweet)
    rows[name] = evaluate(ANN.tweet.tolist(), G, ANN_PRED[name])
ANN_EVAL = pd.DataFrame(rows).T
display(ANN_EVAL.round(3))

pd.set_option("display.max_colwidth", 70)
show = ANN[["tweet", "sarcastic", "phrase", "situation"]].copy()
for name, pr in ANN_PRED.items():
    k = name.split(" · ")[0]
    show[f"{k}: phrase"] = [p["phrase"] for p in pr]
    show[f"{k}: situation"] = [p["situation"] for p in pr]
display(show.head(25))

## 5 · Segment the whole iSarcasmEval test set (for 06b)

1400 tweets (200 sarcastic) and the 200 pairs of task C (sarcastic tweet / the author's non-sarcastic rephrase).

In [ ]:
TEST, PAIRS, TRAIN = load_isarcasm(ISARC)
print("test:", TEST.shape, "| sarcastic:", int(TEST.sarcastic.sum()), "| pairs:", PAIRS.shape, "| train:", TRAIN.shape)

def segment_all(texts):
    out = pd.DataFrame({"text": list(texts)})
    for name in SEG:
        k = name.split(" · ")[0]
        pr = run(name, out.text)
        out[f"{k}_phrase"] = [p["phrase"] for p in pr]
        out[f"{k}_situation"] = [p["situation"] for p in pr]
    return out

t0 = time.time()
SEG_TEST = cached("segments_test", lambda: segment_all(TEST.text), sub=SUB)
SEG_PAIRS0 = cached("segments_pairs_text0", lambda: segment_all(PAIRS.text_0), sub=SUB)
SEG_PAIRS1 = cached("segments_pairs_text1", lambda: segment_all(PAIRS.text_1), sub=SUB)
print(f"{time.time() - t0:.0f} s")

# coverage: share of tweets where both parts are found, by label and by category
rows = []
for k in ["a", "b", "c"]:
    both = SEG_TEST[f"{k}_phrase"].notna() & SEG_TEST[f"{k}_situation"].notna()
    r = {"segmenter": k, "all": both.mean(), "sarcastic": both[TEST.sarcastic == 1].mean(),
         "not sarcastic": both[TEST.sarcastic == 0].mean()}
    for c in CATEGORIES:
        r[c] = both[TEST[c] == 1].mean()
    rows.append(r)
COV = pd.DataFrame(rows).set_index("segmenter")
print("share of tweets with both parts found:")
display(COV.round(3))
print("category counts among the sarcastic tweets:", {c: int(TEST[c].sum()) for c in CATEGORIES})

## 6 · Your own sentences

In [ ]:
OWN = ["Love waiting 2 hours for the bus",
       "Thanks for the flat tire this morning, really made my day",
       "Monday morning meetings, just what I needed",
       "Oh great, another Monday"]
for t in OWN:
    print(t)
    for name in SEG:
        p = run(name, [t])[0]
        print(f"   {name:22s} phrase: {p['phrase']!s:35s} situation: {p['situation']}")

## Save

In [ ]:
d = results_dir(SUB)
SYN_EVAL.to_csv(os.path.join(d, "eval_synthetic.csv"))
ANN_EVAL.to_csv(os.path.join(d, "eval_annotated.csv"))
show.to_csv(os.path.join(d, "annotated_predictions.csv"), index=False)
COV.to_csv(os.path.join(d, "coverage.csv"))
SEG_TEST.to_csv(os.path.join(d, "segments_test.csv"), index=False)
print("saved to", d)